# Inimloetavad ja kontrollitavad diagnostikakaardid

Kaardil on prognoosi kasutaja ja mudeli arendaja vaade. Deterministlik kaart seob prognoosi, vea, stsenaariumid, prognoosi SHAP-panused ja varasemad analoogid. Kaart esitab hüpoteesi, mitte tuvastatud veapõhjuse.

In [ ]:
from pathlib import Path
from functools import partial
import sys
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_run

RUN = latest_run()
table = partial(show_table, RUN, limit=30)
show_run(RUN)

In [ ]:
cards = [read_json(p) for p in sorted((RUN / "cards").glob("card_*/evidence.json"))]
counts = pd.Series([c["diagnosis"]["status"] for c in cards]).value_counts()
display(
    Markdown(
        "Kaartide jaotus: "
        + "; ".join(f"{k}: {v}" for k, v in counts.items())
        + ". Muudatust ei pea soovitama, kui selleks pole piisavalt infot."
    )
)

In [ ]:
table(
    "cards/audit.csv",
    columns=[
        "target_time",
        "all_pass",
        "numeric_error_correct",
        "shap_additive",
        "history_available",
    ],
    rename={
        "numeric_error_correct": "Vea arvutus õige",
        "shap_additive": "SHAP liidetav",
        "history_available": "Ajalugu õigel ajal teada",
    },
)
display(
    Markdown(
        "Tabel näitab, kas kaartide arvutused klapivad ja varasemate juhtumite andmed olid õigel ajal teada. See ei näita, kas inimene kaarti mõistab või muudatus prognoosi parandab."
    )
)

## Sama juhtum kasutajale ja arendajale

Mõlemad vaated kasutavad samu lähteandmeid. Kasutaja näeb lühikest selgitust ja järgmist sammu. Arendaja näeb lisaks kõiki tunnuseid, nende panuseid ja katseplaani.

In [ ]:
card_run = Path(read_json(ROOT / "results/latest_card_study.json")["path"])
index = pd.read_csv(card_run / "card_index.csv")
case = index.loc[index.has_hypothesis].iloc[0]
for role in ("user", "developer"):
    path = (card_run / case[f"{role}_card"]).resolve()
    content = path.read_text(encoding="utf-8")
    evidence_link = "../" + (path.parent / "evidence.json").relative_to(ROOT).as_posix()
    display(Markdown(content.replace("(evidence.json)", f"({evidence_link})")))

## Prognoosi muudatus ja mõõdetud kasu eraldi

Uue kaardi üks arvveerg näitab parandusosa panust prognoosi muutusse, teine sama juhtumi absoluutvea vähenemise Shapley-jaotust. Positiivne prognoosipanus võib saada negatiivse kasujaotuse. Need on tagasivaatelised arvud, mitte vea põhjus ega tulevase edu tõenäosus. Näide valitakse algmudeli suurima absoluutvea järgi. See kaart ei kuulu olemasoleva 12 kaardi piloodi materjalidesse.

In [ ]:
from src.common import read_artifact_csv
from src.card_generator import make_repair_audit
from src.card_views import render_repair_audit

path = RUN / 'refinements/test_attribution_estonia.csv'
if path.exists():
    predictions = read_artifact_csv(path)
    case = (predictions.prediction_GBR - predictions.target_next_hour).abs().idxmax()
    display(Markdown(render_repair_audit(make_repair_audit(predictions.loc[case]))))
else:
    display(Markdown('Arvuta esmalt paranduskasu katse ja Eesti ülekandekatse README juhendi järgi.'))